In [7]:
#import tweepy
import snscrape.modules.twitter as stwt
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm
from multiprocessing.pool import Pool

In [8]:
warnings.filterwarnings("ignore")

In [9]:
from datetime import timedelta
import datetime
from dateutil.relativedelta import relativedelta

In [10]:
def tweet_extraction(week_dates):
    for week in tqdm(week_dates.keys()):
        asset = 'doge'
        week_start = week_dates[week][0]
        week_end = week_dates[week][1]
        tweets_pd = pd.DataFrame(columns =["id", "date", "content", "user_name", "profile_verified_status", "reply_count", "retweet_count",\
             "likes_count", "retweetedTweet", "quotedTweet", "hashtags", "cashtags", "place"])
        start = week_start - timedelta(days=1)
        end = week_start
        while end <= week_end:
            query = f'(Doge OR MultiDodge OR DogCoin OR DogeCoin) lang:en until:{end} since:{start} -filter:replies'
            count = 1
            for tweet in stwt.TwitterSearchScraper(query).get_items():
                #print(vars(tweet))
                _id = tweet.id
                date = tweet.date
                content = tweet.content
                user_name = tweet.user.username
                prof_vstatus = tweet.user.verified
                reply_count = tweet.replyCount
                retweet_count = tweet.retweetCount
                likes_count = tweet.likeCount
                retweetedTweet = tweet.retweetedTweet
                quotedTweet = tweet.quotedTweet
                hashtags = tweet.hashtags
                place = tweet.place
                cashtags = tweet.cashtags
                tweets_pd = tweets_pd.append({"id": _id, "date": date, "content": content, "user_name": user_name,\
                      "profile_verified_status": prof_vstatus, "reply_count": reply_count,\
                      "retweet_count": retweet_count,"likes_count": likes_count, "retweetedTweet": retweetedTweet,
                      "quotedTweet": quotedTweet, "hashtags": hashtags, "cashtags": cashtags, "place": place}, ignore_index= True)
                if count == 1500:
                    break
                count +=1
            start = end
            end += timedelta(days=1)
        dt = datetime.datetime.now()
        dt_string = dt.strftime("%d_%m_%Y_%H_%M_%S")
        name = "tweets_"+asset+'_'+"week_"+str(week)+"_"+dt_string+".csv"
        tweets_pd.to_csv(name)

In [ ]:
def main():
    week_dates={}
    year = 2021
    for week in tqdm(range(1, 52)):
        date = datetime.date(year, 1, 1) + relativedelta(weeks=+week)
        start = date - timedelta(days=date.weekday())
        end = start + timedelta(days=6)
        week_dates[week]= [start, end]
    download = partial(tweet_extraction)
    with Pool(4) as p:
        p.map(download, week_dates)
        

In [ ]:
if __name__ == '__main__':
    main()